# 과제 3: SageMaker 파이프라인 생성
이 과제에서는 [SageMaker Pipelines](https://aws.amazon.com/sagemaker/pipelines/)를 사용하여 엔드투엔드 ML 워크플로우를 생성합니다.

이 과제의 코드 스니펫과 일반적인 지침은 [`03-sagemaker-pipeline.ipynb`](../03-sagemaker-pipeline.ipynb) 노트북을 참조하세요.

## 패키지 임포트

In [ ]:
import pandas as pd
import json
import boto3
import pathlib
import io
import sagemaker
from time import gmtime, strftime, sleep
from sagemaker.deserializers import CSVDeserializer
from sagemaker.serializers import CSVSerializer

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.xgboost.estimator import XGBoost
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import (
    ProcessingInput, 
    ProcessingOutput, 
    ScriptProcessor
)
from sagemaker.inputs import TrainingInput

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import (
    ProcessingStep, 
    TrainingStep, 
    CreateModelStep
)
from sagemaker.workflow.check_job_config import CheckJobConfig
from sagemaker.workflow.parameters import (
    ParameterInteger, 
    ParameterFloat, 
    ParameterString, 
    ParameterBoolean
)
from sagemaker.workflow.clarify_check_step import (
    ModelBiasCheckConfig, 
    ClarifyCheckStep, 
    ModelExplainabilityCheckConfig
)
from sagemaker import Model
from sagemaker.inputs import CreateModelInput
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.conditions import (
    ConditionGreaterThan,
    ConditionGreaterThanOrEqualTo
)
from sagemaker.workflow.pipeline_experiment_config import PipelineExperimentConfig
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import (
    Join,
    JsonGet
)
from sagemaker.workflow.lambda_step import (
    LambdaStep,
    LambdaOutput,
    LambdaOutputTypeEnum,
)
from sagemaker.lambda_helper import Lambda

from sagemaker.model_metrics import (
    MetricsSource, 
    ModelMetrics, 
    FileSource
)
from sagemaker.drift_check_baselines import DriftCheckBaselines

from sagemaker.image_uris import retrieve

sagemaker.__version__

## 상수 설정
코딩을 단순화하기 위해 몇 가지 고정된 문자열 리터럴이 필요합니다. 여기에서 이러한 리터럴을 생성합니다.

In [ ]:
# 파이프라인 객체의 이름 설정
project = "from-idea-to-prod"

pipeline_name = f"{project}-pipeline"
pipeline_model_name = f"{project}-model-xgb"
model_package_group_name = f"{project}-model-group"
endpoint_config_name = f"{project}-endpoint-config"
endpoint_name = f"{project}-endpoint"

# 인스턴스 유형 및 개수 설정
process_instance_type = "ml.c5.xlarge"
train_instance_count = 1
train_instance_type = "ml.m5.xlarge"

# 처리된 데이터에 대한 S3 URL 설정
# train_s3_url = 
# validation_s3_url = 
# test_s3_url = 
# baseline_s3_url = 
# evaluation_s3_url = 

## 연습 1: 파이프라인 생성
ML 워크플로우로 SageMaker 파이프라인을 생성하려면 다음 단계를 따르세요:
- 파이프라인 [매개변수](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#parameters) 설정
- 파이프라인 [단계](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#steps) 구축
- [파이프라인](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.pipeline.Pipeline) 구성
- 파이프라인 [업서트](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.pipeline.Pipeline.upsert)

각 파이프라인 단계에 대한 출력 및 입력을 구성합니다.

In [ ]:
# 파이프라인 매개변수 설정
# 입력 데이터셋에 대한 S3 URL 설정
# input_s3_url_param = ParameterString()


In [ ]:
# 처리 단계
# sklearn_processor = SKLearnProcessor()

# processing_inputs=[]

# processing_outputs=[]

# processor_args = sklearn_processor.run()

# step_process = ProcessingStep()

In [ ]:
# 훈련 단계
# estimator = sagemaker.estimator.Estimator()

# estimator.set_hyperparameters()

# training_inputs = {}

# training_args = estimator.fit(training_inputs)

# step_train = TrainingStep()

평가 단계를 위한 실행 가능한 Python 스크립트를 생성합니다

In [ ]:
%%writefile evaluation_assignment.py

import json
import pathlib
import pickle as pkl
import tarfile
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import datetime as dt
from sklearn.metrics import roc_curve, auc

if __name__ == "__main__":   
    # 모든 경로는 처리 컨테이너에 대해 로컬입니다
        
    # 모델 tar 파일 읽기

    # 모델 로드
    
    # 테스트 데이터 읽기

    # 예측 실행

    # 예측 평가

    # 평가 보고서 저장

이 스크립트를 사용하여 [ScriptProcessor](https://sagemaker.readthedocs.io/en/stable/api/training/processing.html#sagemaker.processing.ScriptProcessor) 객체를 설정합니다. 평가 스크립트가 메트릭을 출력하는 [PropertyFile](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.properties.PropertyFile)을 [ProcessingStep](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.steps.ProcessingStep) 생성자에 전달하는 것을 잊지 마세요.

In [ ]:
# 평가 단계
# script_processor = ScriptProcessor()

# eval_inputs=[]

# eval_outputs=[]

# eval_args = script_processor.run()

# evaluation_report = PropertyFile()

# step_eval = ProcessingStep()

훈련 단계의 모델 아티팩트와 평가 단계의 속성 파일을 사용하여 [Model](https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model) 및 [ModelMetrics](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_metrics.ModelMetrics) 객체를 생성합니다. 이러한 객체를 사용하여 [ModelStep](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.model_step.ModelStep)을 구성합니다.

In [ ]:
# 등록 단계
# model = Model()

# model_metrics = ModelMetrics()

# register_args = model.register()

# step_register = ModelStep()

모델 성능 메트릭이 지정된 임계값을 충족하지 않으면 파이프라인 실행을 중지하는 [FailStep](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.fail_step.FailStep)을 추가합니다.

In [ ]:
# 실패 단계
# step_fail = FailStep()

[조건](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#conditions) 및 `JsonGet`을 사용하여 조건 및 조건 단계를 구성합니다.

In [ ]:
# 조건 단계
# cond_lte = ConditionGreaterThan()

# step_cond = ConditionStep()

단계 배열을 사용하여 [Pipeline](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.pipeline.Pipeline) 객체를 생성합니다.

In [ ]:
# 파이프라인 생성
# pipeline = Pipeline()

In [ ]:
# 새 파이프라인 생성 또는 기존 파이프라인 업데이트
# pipeline.upsert(role_arn=sm_role)

In [ ]:
# 파이프라인 정의 출력
# pipeline_definition = json.loads(pipeline.describe()['PipelineDefinition'])
# pipeline_definition

## 연습 2: 파이프라인 실행

In [ ]:
# 실행 시작
# execution = pipeline.start()

In [ ]:
# 파이프라인 실행이 완료될 때까지 노트북을 대기하려면 이 호출의 주석을 해제하세요
# 이 파이프라인의 실행 시간은 약 13분입니다
# execution.wait()

In [ ]:
# 실행 단계 목록
# execution.list_steps()

## 과제 4 계속하기
[과제 4](04-assignment-sagemaker-project.ipynb) 노트북으로 이동하세요.